# Projeto Aplicado — Sprint 1 · Base Olist
**Segmentação de Clientes por Valor (RFM e Clusterização) para Otimização de Marketing e Retenção em E-commerce de Moda**

Aluno: Fábio Roberto Gomes Filho · Orientador: Marcos Prochnow · Pós-graduação em Ciência de Dados com IA (XP Educação)

| Sprint 1 | 02/11 – 08/11/2026 |
|---|---|
| R01 | Configurar ambiente Python, repositório GitHub e quadro Trello |
| R02 | Coletar e integrar as tabelas Olist |
| R03 | Limpar e tratar a base Olist |
| R04 | Análise exploratória (EDA) |

> A preparação da base da LLVE (R05) fica em um notebook separado (`PA_Sprint1_LLVE`), para que nenhum dado pessoal apareça neste notebook (LGPD).

---
## R01 · Configuração do ambiente
**Critério de aceite:** ambiente funcional (Drive montado, bibliotecas carregadas, arquivos da Olist localizados) e backlog publicado no Trello.

### 1.1 Montagem do Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 1.2 Parâmetros do projeto
Todos os caminhos e parâmetros ficam centralizados nesta célula (requisito **RT3 – Reaplicabilidade**).

In [2]:
from pathlib import Path

# Caminhos no Google Drive
BASE_DIR    = Path('/content/drive/MyDrive/PROJETO APLICADO CD/PROJETO APLICADO')
DATASET_DIR = BASE_DIR / 'DATASET'        # CSVs originais da Olist (somente leitura)
OUTPUT_DIR  = BASE_DIR / 'PROCESSADOS'    # bases tratadas geradas pelos notebooks
FIG_DIR     = BASE_DIR / 'FIGURAS'        # gráficos exportados para o relatório

# Reprodutibilidade (RT1)
SEED = 42

for pasta in (OUTPUT_DIR, FIG_DIR):
    pasta.mkdir(parents=True, exist_ok=True)

print('Base   :', BASE_DIR.exists(), BASE_DIR)
print('Dataset:', DATASET_DIR.exists(), DATASET_DIR)

Base   : True /content/drive/MyDrive/PROJETO APLICADO CD/PROJETO APLICADO
Dataset: True /content/drive/MyDrive/PROJETO APLICADO CD/PROJETO APLICADO/DATASET


### 1.3 Bibliotecas e versões

In [3]:
import sys, platform, random
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn, scipy

random.seed(SEED)
np.random.seed(SEED)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
sns.set_theme(style='whitegrid', context='notebook')

versoes = pd.DataFrame({
    'componente': ['Python', 'pandas', 'numpy', 'matplotlib', 'seaborn', 'scikit-learn', 'scipy'],
    'versão': [platform.python_version(), pd.__version__, np.__version__, matplotlib.__version__,
               sns.__version__, sklearn.__version__, scipy.__version__],
})
versoes

,componente,versão
0,Python,3.13.18
1,pandas,2.2.3
2,numpy,2.4.6
3,matplotlib,3.10.0
4,seaborn,0.13.2
5,scikit-learn,1.6.1
6,scipy,1.16.3


### 1.4 Inventário dos arquivos da Olist
Confere se os 9 arquivos esperados estão na pasta e registra tamanho, linhas e colunas de cada um.

In [4]:
ARQUIVOS_OLIST = {
    'orders':        'olist_orders_dataset.csv',
    'order_items':   'olist_order_items_dataset.csv',
    'payments':      'olist_order_payments_dataset.csv',
    'customers':     'olist_customers_dataset.csv',
    'reviews':       'olist_order_reviews_dataset.csv',
    'products':      'olist_products_dataset.csv',
    'sellers':       'olist_sellers_dataset.csv',
    'geolocation':   'olist_geolocation_dataset.csv',
    'translation':   'product_category_name_translation.csv',
}
USADAS_NO_PROJETO = {'orders', 'order_items', 'payments', 'customers'}   # escopo (Quadro 4)

linhas = []
for nome, arq in ARQUIVOS_OLIST.items():
    caminho = DATASET_DIR / arq
    if caminho.exists():
        df_tmp = pd.read_csv(caminho)
        linhas.append({'tabela': nome, 'arquivo': arq,
                       'tamanho_MB': caminho.stat().st_size / 1e6,
                       'linhas': len(df_tmp), 'colunas': df_tmp.shape[1],
                       'usada_no_projeto': nome in USADAS_NO_PROJETO})
    else:
        linhas.append({'tabela': nome, 'arquivo': arq, 'tamanho_MB': None,
                       'linhas': None, 'colunas': None, 'usada_no_projeto': nome in USADAS_NO_PROJETO})

inventario = pd.DataFrame(linhas).astype({'linhas': 'Int64', 'colunas': 'Int64'})
faltando = inventario[inventario['linhas'].isna()]['arquivo'].tolist()
print('Arquivos encontrados:', len(inventario) - len(faltando), 'de', len(inventario))
if faltando:
    print('⚠️ Faltando:', faltando)
inventario

Arquivos encontrados: 9 de 9


,tabela,arquivo,tamanho_MB,linhas,colunas,usada_no_projeto
0,orders,olist_orders_dataset.csv,17.65,99441,8,True
1,order_items,olist_order_items_dataset.csv,15.44,112650,7,True
2,payments,olist_order_payments_dataset.csv,5.78,103886,5,True
3,customers,olist_customers_dataset.csv,9.03,99441,5,True
4,reviews,olist_order_reviews_dataset.csv,14.45,99224,7,False
5,products,olist_products_dataset.csv,2.38,32951,9,False
6,sellers,olist_sellers_dataset.csv,0.17,3095,4,False
7,geolocation,olist_geolocation_dataset.csv,61.27,1000163,5,False
8,translation,product_category_name_translation.csv,0.00,71,2,False


### 1.5 Checklist do R01
- [x] Google Drive montado e pastas `PROCESSADOS` e `FIGURAS` criadas
- [x] Bibliotecas carregadas e versões registradas (RT1)
- [x] Arquivos da Olist localizados e inventariados
- [ ] Repositório GitHub criado e este notebook salvo nele (*Arquivo → Salvar uma cópia no GitHub*)
- [ ] Quadro Trello com o backlog R01–R16 publicado